In [1]:
import json
import math
import pickle
import os
 
import numpy as np
import pandas as pd
import xgboost as xgb

In [6]:
_DIR = r"C:\Users\maxle\UCInsure\src\hurricane_model"
 
_MODEL_PATH       = os.path.join(_DIR, "hurricane_severity_model_final.json")
_ENCODERS_PATH    = os.path.join(_DIR, "label_encoders_final.pkl")
_LAMBDA_PATH      = os.path.join(_DIR, "lambda_grid.parquet")
_PERCENTILE_PATH  = os.path.join(_DIR, "percentile_lookup.parquet")
_SCALAR_PATH      = os.path.join(_DIR, "calibration_scalar.json")
_METADATA_PATH    = os.path.join(_DIR, "model_metadata.json")

In [8]:
print("[inference] Loading artifacts...")
 
_model = xgb.XGBRegressor()
_model.load_model(_MODEL_PATH)
 
with open(_ENCODERS_PATH, "rb") as f:
    _label_encoders = pickle.load(f)
 
_lambda_df = pd.read_parquet(_LAMBDA_PATH)
_lambda_lookup = {
    (round(lat, 6), round(lon, 6)): (lam, flag)
    for lat, lon, lam, flag in zip(
        _lambda_df['lat'],
        _lambda_df['lon'],
        _lambda_df['lambda'],
        _lambda_df['low_data_flag']
    )
}
 
_percentile_df = pd.read_parquet(_PERCENTILE_PATH)
_pct_log_ael   = _percentile_df["log_ael"].values     # 1001 breakpoints
_pct_values    = _percentile_df["percentile"].values  # 0.0 … 100.0
 
with open(_SCALAR_PATH, "r") as f:
    _cal = json.load(f)
_calibration_scalar = _cal["scalar"]
 
with open(_METADATA_PATH, "r") as f:
    _meta = json.load(f)
_ALL_FEATURES = _meta["features"]
 
print(f"[inference] Ready. Model: {_meta['model_version']}  "
      f"Features: {len(_ALL_FEATURES)}  "
      f"Scalar: {_calibration_scalar:.4f}")

[inference] Loading artifacts...
[inference] Ready. Model: v2_tuned_calibrated  Features: 49  Scalar: 1.1119


In [10]:
_GRID_RES = 0.5
 
def _snap_to_grid(lat: float, lon: float, res: float = _GRID_RES):
    return round(round(lat / res) * res, 6), round(round(lon / res) * res, 6)
 
def _lookup_lambda(lat: float, lon: float):
    if math.isnan(lat) or math.isnan(lon):
        return 0.0, True
    key = _snap_to_grid(lat, lon)
    return _lambda_lookup.get(key, (0.0, True))
 
 
def _encode_features(features: dict) -> pd.DataFrame:
    row = {}
    for col in _ALL_FEATURES:
        val = features.get(col, None)
        if col in _label_encoders:
            le   = _label_encoders[col]
            sval = str(val) if val is not None else "UNKNOWN"
            if sval not in le.classes_:
                sval = "UNKNOWN" if "UNKNOWN" in le.classes_ else le.classes_[0]
            row[col] = int(le.transform([sval])[0])
        else:
            row[col] = float(val) if val is not None else 0.0
    return pd.DataFrame([row], columns=_ALL_FEATURES)
 
 
def _log_ael_to_score(log_ael: float) -> float:
    pct = float(np.interp(log_ael, _pct_log_ael, _pct_values))
    score = 1.0 + (pct / 100.0) * 9.0
    return round(min(max(score, 1.0), 10.0), 2)

In [11]:
def predict(lat: float, lon: float, features: dict) -> dict:
    lam, low_data = _lookup_lambda(lat, lon)
 
    X = _encode_features(features)
    raw_pred = float(_model.predict(X)[0])
 
    ael = raw_pred * lam * _calibration_scalar
 
    if lam == 0.0:
        score = 1.0
    else:
        log_ael = math.log1p(ael)
        score   = _log_ael_to_score(log_ael)
 
    result = {
        "annual_expected_loss": round(ael, 2),
        "risk_score":           score,
        "low_data_flag":        bool(low_data),
        "low_data_note": (
            "Limited historical storm data for this area. "
            "Risk may be lower than average or data coverage is insufficient."
            if low_data else None
        ),
    }
 
    return result